# Notebook 5 — adaptive AC-MPC with the RDP

**Establishes.** Whether estimating the disturbance beats averaging over it.

**Consumes.** `acmpc/model.pkl`, `domrand/DR-all.pkl`.

**Produces.** `acmpc_adaptive/{Base,Robust,Oracle}.pkl`, `variants/{A,B,C}.pkl`,
`estimators/all.pkl`, `nb5_{estimators,scenarios,checks}.csv`, figures F16–F20,
tables T7–T9.

**Decoupled construction — the reason this is attributable.**
1. Train an **Oracle** given the true disturbance. That is the ceiling: what is
   available if estimation were perfect.
2. Replace the oracle with the **RDP**. The oracle→RDP gap is the *cost of
   estimation*; the blind→oracle gap is the *value of the information*.

In [ ]:
import _nbinit  # noqa: F401
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

import adaptive_core_jax as A
import x500_core_jax as X
import study_prelude as S
import study_moderate as M
import viz as V

S.header("Notebook 5 - adaptive AC-MPC with the RDP")
X.style()
FIG = V.figures_dir()

#: **Training is POSITION HOLD.**  arXiv:2605.16015 trains the adaptive policy
#: and the RDP on a position-hold objective and argues that is what "promotes
#: stable, aggressive recovery from severe disturbances while naturally
#: generalizing to dynamic trajectory tracking".  Training on a moving reference
#: instead -- as this notebook previously did -- mixes the tracking error with
#: the disturbance response, which is precisely the signal the RDP has to
#: isolate.  Evaluation stays on the tracking suites, so the generalisation
#: claim is tested rather than assumed.
TRAIN_TASK = "stabilize"
EVAL_TASK = "track"
PATHS = ("circle", "fig8")          # the hold setpoint is sampled from these
CFG = dict(N=1, rep="diag", n_iter=S.CFG["ilqr"], n_diff=2, hid=S.CFG["hid"],
           minib=S.CFG["minib"], epochs=S.CFG["epochs"], sigma=0.05)
ITERS = S.CFG["iters_sweep"]
S2 = M.moderate(S.disturbed_spec(), wind=(0.0, 2.0))
H = A.H_DEFAULT                      # 64, aligned with the paper (C5)
#: How (4.12) converts the moment block.  Under the 17-state control model the
#: conversion is EXACT -- the model has a torque input, so tau maps to angular
#: acceleration by J_nom^-1 -- and 'none' survives only as an ablation.  The
#: 10-state model's (4.13), and its 'first_order'/'closed_loop' modes, are gone:
#: section 2.5 below now measures om_ratio ~ 1 rather than the 69-103x shortfall
#: (4.13) carried.
DMOD_MODE = "exact"

## Precondition gate — checked before any closed-loop number is read

The decomposition is meaningless if the oracle is not *trainable*. A policy
that spends training pinned against its input box never learns a
disturbance-response manifold, and the oracle signal is then correct and
useless.

**Wrench randomisation starts at 0.10 of vehicle weight, not 0.32.** 0.32 is
32 % of weight and saturates training.

In [ ]:
print(f"  training objective: {TRAIN_TASK} (position hold); evaluation: {EVAL_TASK}")
S.section(1, "precondition gate", "is the oracle trainable at all?",
          produces="acmpc_adaptive/{Base,Robust,Oracle}.pkl, T9, F20")

F_DR, T_DR = A.wrench_dr_magnitudes(A.WRENCH_DR_START)
print(f"  wrench randomisation at {A.WRENCH_DR_START:g} of weight: "
      f"F_DR = {F_DR:.3f} N, tau_DR = {T_DR:.4f} N.m")
print(f"  (the top of the sweep, {A.WRENCH_DR_SWEEP[-1]:g}, would be "
      f"F_DR = {A.wrench_dr_magnitudes(A.WRENCH_DR_SWEEP[-1])[0]:.3f} N -- "
      f"do not start there)")

ARMS = {
    "Base":   dict(spec=S.nominal_spec(speed=(1.0, 1.5)), wrench_dr=False, oracle=False),
    "Robust": dict(spec=S2, wrench_dr=A.WRENCH_DR_START, oracle=False),
    "Oracle": dict(spec=S2, wrench_dr=A.WRENCH_DR_START, oracle=True),
}
POL, LOGS = {}, {}
for name, a in ARMS.items():
    env = A.AdaptEnv(S.CFG["n_env"], 5, 200, a["spec"], "central", 0.0,
                     oracle=a["oracle"], wrench_dr=a["wrench_dr"], paths=PATHS,
                     task=TRAIN_TASK)
    actor, critic, log = X.train_ppo(
        None, dict(CFG, dist_label=f"{name} @ wrench_dr={a['wrench_dr']}"),
        seed=5, iters=ITERS, T_rollout=S.CFG["T_rollout"], env=env, verbose=False)
    POL[name], LOGS[name] = actor, log
    X.save_ckpt(dict(actor=actor, critic=critic, cfg=CFG, arm=name,
                     oracle=a["oracle"]), "acmpc_adaptive", f"{name}.pkl")

tail = lambda df: df.iloc[int(0.8 * len(df)):]
# G1: `landed` is the FIRST thing this gate has to check.  A policy that never
# took a step cannot have learned a disturbance-response manifold whatever its
# saturation says, so the gate now fails on it too.
T9 = pd.DataFrame([dict(policy=n, final_reward=float(tail(LOGS[n]).reward.mean()),
                        train_sat=float(tail(LOGS[n]).sat.mean()),
                        crash_rate=float(tail(LOGS[n]).crash_rate.mean()),
                        landed=(int(LOGS[n].landed.sum())
                                if "landed" in LOGS[n] else -1),
                        passed=bool(tail(LOGS[n]).sat.mean() <= 0.05
                                    and ("landed" not in LOGS[n]
                                         or int(LOGS[n].landed.sum()) > 0)))
                   for n in ARMS])
S.table(T9, "T9  training-health gate (pass at 5 % collective saturation)",
        note="a policy pinned against its input box never learns a "
             "disturbance-response manifold; the oracle signal is then correct "
             "and useless",
        csv=("acmpc_adaptive", "nb5_gate.csv"))
GATE_OK = bool(T9.passed.all())
if "landed" in T9 and (T9.landed == 0).any():
    print("!" * 78)
    print("!! G1: the trust region accepted ZERO policy steps for "
          f"{list(T9[T9.landed == 0].policy)}.  Those actors ARE their "
          "initialisation, so Base/Robust/Oracle are the same object up to "
          "their observation width and every gap below is rollout noise.  "
          "Read docs/AUDIT.md G1 first.")
    print("!" * 78)
if not GATE_OK:
    print("!" * 78)
    print("!! PRECONDITION GATE FAILED.  Reduce the wrench randomisation and "
          "retrain.  Moderating the EVALUATION cannot repair a policy trained "
          "against the stop -- the closed-loop numbers below are not readable "
          "as a decomposition into 'value of information' and 'cost of "
          "estimation'.")
    print("!" * 78)
else:
    print(f"\n  Gate PASSED: max training saturation {T9.train_sat.max():.4f} "
          f"<= 0.05.  The oracle is trainable, so the two gaps below are "
          f"attributable.")

fig, ax = plt.subplots(figsize=(6.0, 3.5))
ax2 = ax.twinx()
for i, n in enumerate(ARMS):
    ax.plot(LOGS[n].iter, LOGS[n].reward, color=f"C{i}", lw=1.3, label=n)
    ax2.plot(LOGS[n].iter, LOGS[n].sat, ":", color=f"C{i}", lw=1.1, alpha=0.8)
ax2.axhline(0.05, color="C3", lw=1.2)
ax2.text(0, 0.052, "5 % gate", color="C3", fontsize=7)
ax.set_xlabel("PPO iteration"); ax.set_ylabel("mean reward")
ax2.set_ylabel("collective saturation (dotted)")
ax.legend(fontsize=7, loc="lower right")
ax.set_title("F20  training curves with the precondition gate made visible")
fig.savefig(f"{FIG}/F20_training_gate.png", bbox_inches="tight")
pd.concat([LOGS[n].assign(policy=n) for n in ARMS]).to_csv(
    f"{FIG}/F20_training_gate.csv", index=False)

## Predictor study

Four encoders on the **identical window and target**, so the comparison
isolates the encoder.

> Admissibility is decided by **latency against the 20 ms period**, not by
> $R^2$. A predictor at 137 ms with $R^2=0.88$ is not deployable; one at 2 ms
> with $R^2=0.82$ is.

In [ ]:
S.section(2, "predictor study", "four encoders, identical window and target",
          produces="acmpc_adaptive/estimators/all.pkl, T7, F16, F17")

N_EP_DATA = 12 if S.SCALE == "smoke" else 48
T_DATA = 200 if S.SCALE == "smoke" else 600
pilot = X.make_nmpc_ctrl(N=1, n_iter=S.CFG["ilqr"])

def collect(scen, level, seed, n, T):
    """Fly under a known disturbance and log the causal frame and the truth.

    The window is built from the environment's ring buffer, which the
    disturbance manager cannot write to (§9.1): no injected disturbance value
    can reach the predictor's input.
    """
    env = A.AdaptEnv(n, seed, T + 10, S2, scen, level, paths=PATHS,
                     wrench_dr=A.WRENCH_DR_START, task=TRAIN_TASK, H=H)
    pilot.bind_env(env)
    F, Y, EP = [], [], []
    o, e, xr = env.obs()
    for k in range(T):
        _, uref, _ = env.ref_now()
        u, _ = pilot(o, e, xr, uref=uref)
        F.append(np.asarray(env.frame26()))
        Y.append(np.asarray(env.d_truth()))
        EP.append(np.asarray(env.n_step))
        env.step(u)
        o, e, xr = env.obs()
    return np.array(F), np.array(Y), np.array(EP)

print(f"  generating data: H={H}, {N_EP_DATA} vehicles x {T_DATA} steps per scenario")
Ws, Ys = [], []
for scen, lvl in (("central", 0.15), ("asym", 0.07), ("slung", 10.0)):
    f, y, ep = collect(scen, lvl, 31, N_EP_DATA, T_DATA)
    # episode id per (t, vehicle): a respawn resets n_step, so a decreasing
    # n_step marks a boundary.  Windows must never straddle one (§9.6).
    eid = np.cumsum(np.diff(ep, axis=0, prepend=ep[:1]) < 0, axis=0)
    w, yy = A.make_windows(f, y, H, ep_id=eid)
    Ws.append(w); Ys.append(yy)
W = np.concatenate(Ws); Y = np.concatenate(Ys)
print(f"  windows: {W.shape}, targets: {Y.shape}")

# split BY EPISODE, never by shuffled samples
n_all = W.shape[0]
rng = np.random.default_rng(0)
blocks = np.arange(n_all) // max(T_DATA - H, 1)
ub = np.unique(blocks)
tr_b, va_b, te_b = A.split_by_episode(len(ub), seed=0)
m_tr = np.isin(blocks, ub[tr_b]); m_va = np.isin(blocks, ub[va_b])
m_te = np.isin(blocks, ub[te_b])
print(f"  split by complete episode: train {m_tr.sum()}, val {m_va.sum()}, "
      f"test {m_te.sum()}")

rows, PARAMS = [], {}
for kind in A.ENCODERS:
    p, sc, hist = A.train_rdp(jax.random.PRNGKey(7), kind, W[m_tr], Y[m_tr],
                              W[m_va], Y[m_va], H=H, epochs=S.CFG["est_epochs"],
                              batch=S.CFG["est_batch"], verbose=False)
    pred = A.rdp_predict(p, sc, W[m_te])
    r2 = A.r2_score(Y[m_te], pred)
    lat = A.rdp_latency_ms(p, sc, H)
    rows.append(dict(encoder=kind, r2_overall=float(np.nanmean(r2)),
                     r2_force=float(np.nanmean(r2[:3])),
                     r2_moment=float(np.nanmean(r2[3:])),
                     params=A.rdp_param_count(p), lat_median_ms=lat["median"],
                     lat_p95_ms=lat["p95"],
                     admissible=bool(lat["p95"] < 20.0),
                     **{f"r2_{c}": float(v) for c, v in zip(A.CHANNEL_NAMES, r2)}))
    PARAMS[kind] = (p, sc, r2)
T7 = pd.DataFrame(rows)
S.table(T7[["encoder", "r2_overall", "r2_force", "r2_moment", "params",
            "lat_median_ms", "lat_p95_ms", "admissible"]],
        "T7  predictor accuracy and cost",
        note="admissibility is decided by latency against the 20 ms period, "
             "not by R^2",
        csv=("acmpc_adaptive", "nb5_estimators.csv"))

adm = T7[T7.admissible]
SELECTED = (adm.sort_values("r2_overall", ascending=False).encoder.iloc[0]
            if len(adm) else T7.sort_values("r2_overall", ascending=False).encoder.iloc[0])
print(f"\n  SHIP: {SELECTED}.  Chosen among the admissible encoders "
      f"({list(adm.encoder)}) by R^2, because latency is a hard constraint and "
      f"accuracy is the objective -- not the other way round.")
ck = {}
for kind in A.ENCODERS:
    p, sc, r2 = PARAMS[kind]
    ck.update(A.ckpt_entry(kind, p, kind, dict(H=H, hid=(128, 64)), sc["mu"],
                           sc["sd"], sc["out_sd"], H, r2))
ck["__selected__"] = SELECTED
X.save_ckpt(ck, "acmpc_adaptive", "estimators", "all.pkl")

fig, ax = plt.subplots(figsize=(5.8, 3.8))
for i, r in T7.iterrows():
    ax.scatter(r.lat_p95_ms, r.r2_overall, s=30 + r.params / 400.0,
               marker=["o", "s", "^", "v"][i], label=f"{r.encoder} ({r.params/1e3:.0f}k)")
ax.axvline(20.0, color="C3", lw=1.4)
ax.text(20.5, ax.get_ylim()[0], "20 ms deadline", color="C3", fontsize=7, rotation=90)
ax.set_xscale("log"); ax.set_xlabel("single-window p95 latency [ms] (log)")
ax.set_ylabel(r"test $R^2$ (mean over 6 channels)")
ax.legend(fontsize=7); ax.set_title("F16  accuracy-latency trade; size = parameters")
fig.savefig(f"{FIG}/F16_encoder_trade.png", bbox_inches="tight")
T7.to_csv(f"{FIG}/F16_encoder_trade.csv", index=False)

fig, ax = plt.subplots(figsize=(6.0, 3.3))
r2s = PARAMS[SELECTED][2]
ax.bar(range(6), r2s, color=["C0"] * 3 + ["C1"] * 3, edgecolor="k", lw=0.5)
ax.axhline(0.0, color="C3", lw=1.4)
ax.text(5.2, 0.02, "break-even", color="C3", fontsize=7, ha="right")
ax.set_xticks(range(6)); ax.set_xticklabels(A.CHANNEL_NAMES)
ax.set_ylabel(r"test $R^2$"); ax.set_title(f"F17  per-channel $R^2$, {SELECTED}")
fig.savefig(f"{FIG}/F17_channel_r2.png", bbox_inches="tight")
pd.DataFrame(dict(channel=A.CHANNEL_NAMES, r2=r2s)).to_csv(
    f"{FIG}/F17_channel_r2.csv", index=False)
print(f"  channels below break-even (R^2 <= 0): "
      f"{[c for c, v in zip(A.CHANNEL_NAMES, r2s) if v <= 0] or 'none'}")

# F19 -- true vs predicted over one episode
f1, y1, _ = collect("asym", 0.07, 77, 4, 240)
w1, yt1 = A.make_windows(f1, y1, H)
pv = A.rdp_predict(PARAMS[SELECTED][0], PARAMS[SELECTED][1], w1)
nveh = f1.shape[1]
fig, axs = plt.subplots(2, 3, figsize=(11.5, 4.6), sharex=True)
for i in range(6):
    a_ = axs[i // 3, i % 3]
    a_.plot(yt1[::nveh, i], color="C0", lw=1.2, label="truth")
    a_.plot(pv[::nveh, i], "--", color="C1", lw=1.2, label="RDP")
    a_.set_title(f"{A.CHANNEL_NAMES[i]} [{A.CHANNEL_UNITS[i]}]", fontsize=8)
    if i == 0:
        a_.legend(fontsize=6.5)
fig.suptitle(f"F19  true vs predicted wrench, {SELECTED}, asym f=0.07", fontsize=10)
fig.savefig(f"{FIG}/F19_wrench_timeseries.png", bbox_inches="tight")
pd.DataFrame(np.c_[yt1[::nveh], pv[::nveh]],
             columns=[f"true_{c}" for c in A.CHANNEL_NAMES]
             + [f"pred_{c}" for c in A.CHANNEL_NAMES]).to_csv(
    f"{FIG}/F19_wrench_timeseries.csv", index=False)

## The two disturbance objects, measured against each other

§4.3 defines two different things and §4.4 warns they must not be confused:
the **wrench** (b) in [N, N·m], and the **model residual** (a) in
[m/s², rad/s]. `oracle_target` selects which the observation carries, and it
had never been exercised on `'residual'` (audit G1).

This matters because of D3: (4.13) converts the moment block using the rate
loop's steady state *with $K_i$ neglected*, which is a short-transient model.
The payload scenarios are quasi-static, so the conversion and the true
residual can differ by an order of magnitude. Here both are measured.

In [ ]:
S.section(2.5, "wrench vs residual", "the two disturbance objects, side by side",
          produces="acmpc_adaptive/nb5_dmod_validity.csv")

rows = []
for scen, lvl in (("central", 0.15), ("asym", 0.01), ("asym", 0.07)):
    env = A.AdaptEnv(4, 1, 200000, S.nominal_spec(speed=(0.0, 0.0)), scen, lvl,
                     paths=("hover",), task="stabilize", H=H,
                     fixed=dict(e0=[0.0, 0.0, 0.0]))
    pilot2 = X.make_lqr_ctrl()
    for _ in range(800):
        o, e, xr = env.obs()
        _, uref, _ = env.ref_now()
        u, _ = pilot2(o, e, xr, uref=uref)
        env.step(u)
    truth = np.asarray(env.residual_truth()).mean(0)           # (4.9), the truth
    for mode in X.DMOD_MODES:
        conv = np.asarray(X.wrench_to_dmod(env.d_truth(), mode)).mean(0)
        rows.append(dict(scen=scen, level=lvl, mode=mode,
                         a_res_conv=float(np.linalg.norm(conv[:3])),
                         a_res_true=float(np.linalg.norm(truth[:3])),
                         om_res_conv=float(np.linalg.norm(conv[3:])),
                         om_res_true=float(np.linalg.norm(truth[3:])),
                         om_ratio=float(np.linalg.norm(conv[3:])
                                        / max(np.linalg.norm(truth[3:]), 1e-9))))
DV = pd.DataFrame(rows)
S.table(DV, "(4.12) conversion against the true residual (4.9)",
        note="om_ratio = 1 means the conversion is exact.  Under the 17-state "
             "model BOTH blocks convert exactly: F/m and J_nom^-1 tau",
        csv=("acmpc_adaptive", "nb5_dmod_validity.csv"))
print(f"\n  Both blocks convert exactly: a_res = F/m_nom and alpha_res = J_nom^-1 tau.")
print(f"  Under the 10-state model the moment block had no exact image -- (4.13)")
print(f"  approximated it by the rate loop's steady state with K_i neglected, which")
print(f"  measured 69-103x SMALLER than the (4.9) residual it was meant to cancel.")
print(f"  The 17-state model has a torque input, so om_ratio should now read ~1.")
print(f"  The model path below uses mode={DMOD_MODE!r}.")

# and the observation channel can carry either object
for tgt in ("wrench", "residual"):
    env = A.AdaptEnv(4, 1, 200000, S.nominal_spec(speed=(0.0, 0.0)), "asym", 0.07,
                     paths=("hover",), task="stabilize", oracle=True,
                     oracle_target=tgt, H=H, fixed=dict(e0=[0.0, 0.0, 0.0]))
    for _ in range(60):
        o, e, xr = env.obs()
        _, uref, _ = env.ref_now()
        u, _ = X.make_lqr_ctrl()(o, e, xr, uref=uref)
        env.step(u)
    blk = np.asarray(env.obs()[0])[:, X.OBS_DIM:]
    print(f"  oracle_target={tgt!r:10s} observation block rms = {np.sqrt((blk**2).mean()):.4f} "
          f"[{'N, N.m' if tgt == 'wrench' else 'm/s^2, rad/s'}]")

## Closed loop — the three variants

| variant | → model | → observation | interpretation |
|---|---|---|---|
| A | no | yes | the cost map may react; model stays nominal |
| B | yes | no | model corrected; cost map stays blind |
| C | yes | yes | both |

**Evaluation levels are moderated to 40 % of the study brackets.** The raw
brackets put a quarter of the vehicle mass off-centre, which on a 1.689
thrust-to-weight airframe sits against the actuator limit — and for `asym`,
above the rate-loop trim limit entirely (N-7).

**Test only what you trained.** The checks below are built from the variant
names that actually exist; a filter on an absent name returns `NaN`, and
`NaN <= x` is `False`, so every check would report failure for a reason
unrelated to the controllers.

In [ ]:
S.section(3, "closed loop", "does estimating beat averaging?",
          produces="acmpc_adaptive/variants/*.pkl, nb5_scenarios.csv, T8, F18")

for scen in A.SCENARIOS:
    raw = A.SCEN_LEVELS[scen]
    mod = A.moderate_levels(raw)
    print(f"  {scen:8s} levels  raw {raw}  ->  moderated (40 %) {mod}")
print(f"  asym trim limit (N-7): f_max = "
      f"{X.J_NOM[0,0]*X.P.K_i[0]*X.P.I_lim/(0.174*X.M_TOT*X.P.g):.4f} -- the raw "
      f"top level {A.SCEN_LEVELS['asym'][-1]} exceeds it, the moderated one does not")

VARIANTS = {"A": dict(to_model=False, to_obs=True),
            "B": dict(to_model=True, to_obs=False),
            "C": dict(to_model=True, to_obs=True)}
p_sel, sc_sel, _ = PARAMS[SELECTED]
# G6: a variant whose PREDICTION DYNAMICS carry the residual must be TRAINED
# that way.  `train_ppo` reads the correction only when `use_d` is set, and CFG
# never set it, so B and C used to train on nominal prediction dynamics and were
# then evaluated with `dmod_fn` feeding the residual into fc() -- a train/eval
# mismatch on the one axis the variant study is about.  The estimator is
# attached to the TRAINING env too, so the policy learns against the prediction
# it will be deployed with rather than against ground truth.
for v, spec_v in VARIANTS.items():
    env = A.AdaptEnv(S.CFG["n_env"], 5, 200, S2, "central", 0.0,
                     oracle=spec_v["to_obs"], wrench_dr=A.WRENCH_DR_START,
                     paths=PATHS, task=TRAIN_TASK, H=H, dmod_mode=DMOD_MODE)
    env.attach(p_sel, sc_sel)
    cfg_v = dict(CFG, dist_label=f"variant {v}", use_d=bool(spec_v["to_model"]))
    actor, critic, log = X.train_ppo(None, cfg_v,
                                     seed=5, iters=ITERS,
                                     T_rollout=S.CFG["T_rollout"], env=env,
                                     verbose=False)
    X.save_ckpt(dict(actor=actor, critic=critic, cfg=cfg_v, variant=v, **spec_v),
                "acmpc_adaptive", "variants", f"{v}.pkl")
    VARIANTS[v]["actor"] = actor


#: Which (4.13) mode the MODEL path uses.  D3 measures the closed-loop gain of
#: the rate loop to a step moment: ~0.9 out to 0.3 s but 0.10 by 8 s, because
def adapt_ctrl(actor, name, to_model, attach_rdp):
    cfg = dict(CFG, n_diff=1, name=name)
    dmod_fn = (lambda ev: ev.dmod()) if to_model else None
    return X.ctrl_from_actor(actor, cfg, dmod_fn=dmod_fn)


rows = []
for scen in A.SCENARIOS:
    for lvl in A.moderate_levels(A.SCEN_LEVELS[scen]):
        # G6: `to_model=False` for all three reference arms.  Base, Robust and
        # Oracle are trained with `use_d` unset -- nominal prediction dynamics --
        # so handing Oracle a residual-corrected fc() at evaluation only was a
        # train/eval mismatch, and it is the arm the whole decomposition is
        # measured against.  Oracle's privilege is the OBSERVATION channel.
        entries = [("Base", POL["Base"], False, False),
                   ("Robust", POL["Robust"], False, False),
                   ("Oracle", POL["Oracle"], False, False)]
        entries += [(f"RDP-{v}", VARIANTS[v]["actor"], VARIANTS[v]["to_model"], True)
                    for v in VARIANTS]
        for name, actor, to_model, attach in entries:
            env = A.AdaptEnv(max(S.CFG["n_eval"] // 4, 8), 911, 100000, S2, scen,
                             lvl, oracle=(name == "Oracle"
                                          or name in ("RDP-A", "RDP-C")),
                             paths=PATHS, H=H, task=EVAL_TASK,
                             dmod_mode=DMOD_MODE)
            if attach:
                env.attach(p_sel, sc_sel)
            ctrl = adapt_ctrl(actor, name, to_model, attach)
            st = X.stats(X.rollout_eval(env, ctrl, S.CFG["T_eval"] // 3, warmup=30))
            rows.append(dict(scen=scen, level=lvl, ctrl=name, **st))
RES = pd.DataFrame(rows)
T8 = RES.pivot_table(index=["scen", "level"], columns="ctrl",
                     values="rmse").reset_index()
S.table(T8, "T8  closed-loop RMSE by scenario and level",
        note="the ZERO-level row is the most important one: any gap there is "
             "conservatism, not disturbance rejection",
        csv=("acmpc_adaptive", "nb5_scenarios.csv"))
RES.to_csv(X.apath("acmpc_adaptive", "nb5_scenarios_long.csv"), index=False)

# checks, built only from the variants that exist
RDPS = sorted(n for n in RES.ctrl.unique() if n.startswith("RDP"))
med = lambda sc, n: float(RES[(RES.scen == sc) & (RES.ctrl == n)].rmse.median())
checks = []
for scen in A.SCENARIOS:
    if not RDPS:
        continue
    best = min(med(scen, n) for n in RDPS)
    checks += [(f"{scen}: Oracle <= best RDP", med(scen, "Oracle") <= best + 1e-9),
               (f"{scen}: best RDP <= Robust", best <= med(scen, "Robust")),
               (f"{scen}: best RDP <= Base", best <= med(scen, "Base"))]
CHK = pd.DataFrame(checks, columns=["check", "passed"])
S.table(CHK, "Closed-loop checks (only over the variants that were trained)",
        note=f"variants present: {RDPS}", csv=("acmpc_adaptive", "nb5_checks.csv"))

fig, axs = plt.subplots(1, 3, figsize=(12.5, 3.4))
for j, scen in enumerate(A.SCENARIOS):
    d = RES[RES.scen == scen]
    for i, c in enumerate(sorted(d.ctrl.unique())):
        dd = d[d.ctrl == c].sort_values("level")
        axs[j].plot(dd.level, dd.rmse, "-", color=f"C{i}",
                    marker=["o", "s", "^", "v", "D", "P"][i % 6], ms=4, label=c)
    axs[j].set_title(scen); axs[j].set_xlabel("level (moderated)")
    z = d[d.level == d.level.min()]
    axs[j].axvline(d.level.min(), color="0.5", ls=":", lw=1)
axs[0].set_ylabel("position RMSE [m]"); axs[0].legend(fontsize=6, ncol=2)
fig.suptitle("F18  closed loop vs level -- the ZERO level is the key point",
             fontsize=10)
fig.savefig(f"{FIG}/F18_closed_loop.png", bbox_inches="tight")
RES.to_csv(f"{FIG}/F18_closed_loop.csv", index=False)

## Analysis — gate first, then the two gaps, then the variants

In [ ]:
print("\n  1. GATE.", "passed" if GATE_OK else "FAILED -- everything below is "
      "not readable as a decomposition")
print(f"     max training saturation {T9.train_sat.max():.4f} against the 0.05 limit\n")

print("  2. THE ZERO ROW FIRST.  Any gap at level 0 is conservatism, not "
      "disturbance rejection:")
for scen in A.SCENARIOS:
    z = RES[(RES.scen == scen) & (RES.level == RES[RES.scen == scen].level.min())]
    if len(z):
        base0 = float(z[z.ctrl == "Base"].rmse.iloc[0])
        for c in sorted(z.ctrl.unique()):
            v = float(z[z.ctrl == c].rmse.iloc[0])
            print(f"     {scen:8s} level 0: {c:8s} {v:.4f} m "
                  f"({100*(v-base0)/max(base0,1e-9):+6.1f}% vs Base)")
        break

print("\n  3. THE TWO GAPS (median over levels):")
for scen in A.SCENARIOS:
    if not RDPS:
        continue
    blind, orc = med(scen, "Robust"), med(scen, "Oracle")
    best_rdp = min(med(scen, n) for n in RDPS)
    print(f"     {scen:8s}: value of information (Robust - Oracle) = "
          f"{blind-orc:+.4f} m; cost of estimation (bestRDP - Oracle) = "
          f"{best_rdp-orc:+.4f} m")

print("\n  4. WHICH VARIANT WINS (median over all scenarios and levels):")
for n in RDPS:
    print(f"     {n}: {float(RES[RES.ctrl==n].rmse.median()):.4f} m")
if RDPS:
    win = min(RDPS, key=lambda n: float(RES[RES.ctrl == n].rmse.median()))
    says = {"RDP-A": "the disturbance belongs in the COST, not the model: the "
                     "cost map reacts to it while the prediction dynamics stay "
                     "nominal",
            "RDP-B": "the disturbance belongs in the MODEL: correcting the "
                     "prediction dynamics is enough and the cost map need not "
                     "see it",
            "RDP-C": "the disturbance belongs in BOTH: correcting the model and "
                     "letting the cost map react are complementary"}
    print(f"     -> {win} wins, which says {says[win]}.")
    print(f"     Judge this against the seed spread reported in Notebook 3 "
          f"before calling it a result.")
S.scale_guard(RES.rename(columns={"ctrl": "ctrl"}))

X.save_json(dict(scale=S.SCALE, selected_encoder=SELECTED, gate_passed=GATE_OK,
                 H=H, variants=list(VARIANTS), checks=CHK.to_dict("records")),
            "acmpc_adaptive", "nb5_summary.json")

## What this notebook establishes

- **The gate is reported first** and everything downstream is conditioned on
  it. A saturated oracle makes the decomposition meaningless, and moderating
  the *evaluation* cannot repair a policy trained against the stop.
- The encoder is chosen on **latency**, with $R^2$ as the objective *inside*
  the admissible set — the reverse of the usual ordering, and the one the 20 ms
  period demands.
- The zero-level row is read first, because a gap there is conservatism rather
  than disturbance rejection.
- The two gaps are reported separately: blind→oracle is the *value of the
  information*, oracle→RDP is the *cost of estimating it*. That separation is
  the entire reason the construction is decoupled.

In [ ]:
S.checkpoint(base=("acmpc_adaptive", "Base.pkl"),
             robust=("acmpc_adaptive", "Robust.pkl"),
             oracle=("acmpc_adaptive", "Oracle.pkl"),
             variantA=("acmpc_adaptive", "variants", "A.pkl"),
             estimators=("acmpc_adaptive", "estimators", "all.pkl"),
             scenarios=("acmpc_adaptive", "nb5_scenarios.csv"))
print("\nNotebook 5 complete.")